## Datagen

In [ ]:
# These shell commands work in Jupyter/Colab
# (not in plain Python scripts)

!pip install MDSA-tools mdtraj

!wget -q -O system_one.prmtop https://raw.githubusercontent.com/zeper-eng/MDSA-tools/main/PDBs/5JUP_N2_CGU_nowat.prmtop
!wget -q -O system_one.mdcrd  https://raw.githubusercontent.com/zeper-eng/MDSA-tools/main/PDBs/CCU_CGU_10frames.mdcrd

!wget -q -O system_two.prmtop https://raw.githubusercontent.com/zeper-eng/MDSA-tools/main/PDBs/5JUP_N2_GCU_nowat.prmtop
!wget -q -O system_two.mdcrd  https://raw.githubusercontent.com/zeper-eng/MDSA-tools/main/PDBs/CCU_GCU_10frames.mdcrd



In [4]:

from mdsa_tools.Data_gen_hbond import TrajectoryProcessor as tp
import numpy as np
import os

#load in and test trajectory
system_one_topology = "system_one.prmtop"
system_one_trajectory = "system_one.mdcrd"

system_two_topology = "system_two.prmtop"
system_two_trajectory = "system_two.mdcrd"


test_trajectory_one = tp(trajectory_path=system_one_trajectory,topology_path=system_one_topology)
test_trajectory_two = tp(trajectory_path=system_two_trajectory,topology_path=system_two_topology)


#now that its loaded in try to make object
test_system_one_ = test_trajectory_one.create_system_representations()
test_system_two_ = test_trajectory_two.create_system_representations()


np.save('test_system_one',test_system_one_)
np.save('test_system_two',test_system_two_)


## ANALYSIS

In [8]:
from mdsa_tools.Analysis import systems_analysis

all_systems = [test_system_one_, test_system_two_]
Systems_Analyzer = systems_analysis(all_systems)

# transform adjacency matrices, perform clustering and dimensional reduction
Systems_Analyzer.replicates_to_featurematrix()
optimal_k_silhouette_labels, optimal_k_elbow_labels, centers_silhouette, centers_elbow = Systems_Analyzer.perform_kmeans(outfile_path='.', max_clusters=5)
print('clustering successfully completed')
X_pca, weights, explained_variance_ratio_ = Systems_Analyzer.reduce_systems_representations(method='PCA')  # you could do method='PCA'/'UMAP' here
print('reduction successful')



clustering successfully completed
X_pca shape (new data): (18, 2)
the total explained variance0.5087422728538513
the total explained variance of PC's is [0.4567998  0.05194245]
weights shape: (2, 121771)
reduction successful


In [9]:
import matplotlib.cm as cm
from mdsa_tools.Viz import visualize_reduction
#visualize embedding space with original clusters
visualize_reduction(X_pca,color_mappings=optimal_k_silhouette_labels,savepath='./PCA_',cmap=cm.plasma_r)

In [10]:
#If they exist map transitions between the various cluster assignments
from mdsa_tools.Viz import replicatemap_from_labels

replicatemap_from_labels(cmap=cm.plasma_r,frame_list=[9]*2,labels=optimal_k_silhouette_labels,savepath='./Repmap_')#9 frames each so